# Serve API + Cloudflare Tunnel — chạy local FE, model/DB nạp trên Colab (GPU)

**Bấm Run all rồi đợi ~3-5 phút.** Notebook này KHÔNG chạy ETL và KHÔNG chạy đánh
giá — nó chỉ khởi động server `python main.py --api` (đọc DB đã dựng xong trên
Drive, giống `colab_runtime_eval.ipynb`) rồi mở một **Cloudflare Quick Tunnel**
(`trycloudflare.com`, không cần tài khoản/domain) trỏ vào cổng đó. URL tunnel in
ra ở mục 10 là thứ bạn dán vào FE chạy trên máy local (`project_rag_fe`, xem
`NEXT_PUBLIC_API_HOST` ở mục 11) — nhờ vậy FE ở local gọi được LLM + retrieval
đang chạy trên GPU của Colab mà không cần triển khai gì thêm.

## Trước khi chạy

- **`database/` phải đã được upload lên Drive** dưới dạng `database_png`, cùng
  cấp thư mục `datasource_png` — đúng bản mà `colab_runtime_eval.ipynb` đang
  dùng (D-165). Notebook này chỉ ĐỌC, không ghi gì ngược lại Drive.
- **Chỉ cần `database_png`**, KHÔNG cần `datasource_png` — trang PNG gốc chỉ
  dùng lúc ETL (`RAG_DATA_DIR`); đường serve chỉ đọc `database/images/` (hình đã
  crop, nằm sẵn trong `database_png`) qua route `/images/<path>`, không đọc
  `datasources/` (đã kiểm: không route/`AppServices` nào import `DATA_DIR`).
- Chọn **Runtime → Change runtime type → GPU** trước khi Run all (Qwen2.5-3B chạy
  CPU vẫn được nhưng chậm hẳn cho demo tương tác).
- Cần secret `HF_TOKEN` (mở tab 🔑 bên trái, bật *Notebook access*) — giống hai
  notebook kia.
- Sau khi có URL tunnel, mở `project_rag_fe/.env.local`, đặt
  `NEXT_PUBLIC_API_HOST=<url_tunnel>` rồi `npm run dev` — xem mục 11.

**Lưu ý:** URL `trycloudflare.com` sinh **NGẪU NHIÊN mỗi lần chạy lại** mục 10 —
đổi lại `.env.local` mỗi khi bạn restart notebook.

## 1. Clone repo

In [ ]:
!git clone -b master https://github.com/lcdkhoa/project-bio-rag.git
%cd project-bio-rag

In [ ]:
!git log --oneline -3

## 2. Cài dependencies

Đường serve không OCR (không cần `mineru_vl_utils`/`poppler`/`tesseract` như
notebook ETL) và không gọi LLM giám khảo (không cần `GROQ_API_KEY` như notebook
eval) — chỉ cần đúng `requirements.txt`.

In [ ]:
!pip install -q -r requirements.txt
import transformers
print("transformers:", transformers.__version__)

## 3. Secret + env cơ bản

`HF_TOKEN` lấy từ Colab Secrets, giống hai notebook kia.

In [ ]:
import os, multiprocessing
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

n = multiprocessing.cpu_count()
os.environ["OMP_NUM_THREADS"] = str(n)
os.environ["NUMEXPR_NUM_THREADS"] = str(n)
os.environ["OPENBLAS_NUM_THREADS"] = str(n)
os.environ["USE_GPU"] = "true"
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")), "| CPU cores:", n)

## 4. Tải model về `./models` (chạy ONLINE, trước khi bật offline)

Dùng profile `serve` (`src/utils/download_models.py`) — đúng bốn model cần cho
truy vấn + sinh câu trả lời: `bge-m3` (embedding), `bge-reranker-v2-m3` (rerank,
`RERANK_ENABLED=true` là mặc định), `Qwen2.5-3B-Instruct` (sinh câu trả lời),
`clip-vit-base-patch16` (ảnh — `AppServices` luôn nạp collection ảnh dù câu hỏi
có cần hình hay không).

In [ ]:
import subprocess, sys

r = subprocess.run([sys.executable, "-u", "./src/utils/download_models.py",
                    "--save_dir", "./models", "--profile", "serve"])
if r.returncode != 0:
    raise RuntimeError(
        f"Tai model that bai (ma thoat {r.returncode}) - DUNG o day.")
print("Tai model xong, ma thoat 0.")

In [ ]:
import os

for ten, thu_muc in [("bge-m3", "bge-m3"),
                     ("bge-reranker-v2-m3", "bge-reranker-v2-m3"),
                     ("Qwen2.5-3B-Instruct", "Qwen2.5-3B-Instruct"),
                     ("clip-vit-base-patch16", "clip-vit-base-patch16")]:
    p = f"./models/{thu_muc}"
    con = os.path.exists(f"{p}/config.json")
    print(f"{ten:24s} config.json={con}")
    assert con, f"{ten}: khong thay config.json o {p} - tai model that bai that su"
print("OK - du 4 model.")

## 5. Mount Drive + khôi phục `database_png` (chỉ đọc, copy về đĩa cục bộ)

Copy toàn bộ vào **đĩa cục bộ** `/content/database` chứ không đọc thẳng qua
Drive-FUSE — ChromaDB dùng SQLite, mở qua FUSE dễ lock/hỏng (D-152, cùng lý do
`colab_runtime_etl.ipynb`/`colab_runtime_eval.ipynb` đã áp dụng). `_copy_resilient`
đi từng file, thử lại khi Drive-FUSE rớt kết nối giữa chừng (ENOTCONN, D-161).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import shutil
import time
from pathlib import Path


def _copy_resilient(src: Path, dst: Path, tries: int = 5, delay: float = 5.0):
    """Di tung file, thu lai khi Drive-FUSE rot ket noi (ENOTCONN, D-161) - mot
    file loi khong huy phan cay da sao chep duoc."""
    if src.is_dir():
        dst.mkdir(parents=True, exist_ok=True)
        for child in src.iterdir():
            _copy_resilient(child, dst / child.name, tries, delay)
        return
    for attempt in range(1, tries + 1):
        try:
            shutil.copy2(src, dst)
            return
        except OSError as exc:
            if attempt == tries:
                raise
            print(f"  loi copy {src.name} (lan {attempt}/{tries}): {exc} -> thu lai sau {delay}s")
            time.sleep(delay)


print("OK, dinh nghia xong _copy_resilient")

In [ ]:
import os

DB_SOURCE_DIR = "/content/drive/MyDrive/project_bio_rag/database_png"  # SUA neu khac
os.environ["RAG_DATABASE_DIR"] = "/content/database"

base = "/content/project-bio-rag/models"
os.environ["EMBEDDING_MODEL"] = f"{base}/bge-m3"
os.environ["RERANK_MODEL"] = f"{base}/bge-reranker-v2-m3"
os.environ["LLM_MODEL"] = f"{base}/Qwen2.5-3B-Instruct"
os.environ["CLIP_MODEL"] = f"{base}/clip-vit-base-patch16"
os.environ["HF_HUB_OFFLINE"] = "1"  # model da tai o muc 4

src_dir = Path(DB_SOURCE_DIR)
if not src_dir.exists():
    parent = src_dir.parent
    goi_y = list(parent.iterdir()) if parent.exists() else []
    raise RuntimeError(
        f"Khong thay {src_dir}. Cac thu muc con trong {parent}:\n"
        + "\n".join(str(p) for p in goi_y)
        + "\n-> sua DB_SOURCE_DIR o o tren cho dung, dung doan.")

local_db = Path(os.environ["RAG_DATABASE_DIR"])
local_db.mkdir(parents=True, exist_ok=True)
for item in src_dir.iterdir():
    print("dang khoi phuc:", item.name)
    _copy_resilient(item, local_db / item.name)
print("XONG khoi phuc DB tu Drive.")

## 6. Xác nhận index khôi phục đúng — đừng bỏ qua

Đo trực tiếp trên DB vừa khôi phục, giống mục 8 của `colab_runtime_eval.ipynb`.
Đây là kiểm tra MỀM (in số ra để bạn tự đối chiếu với `CLAUDE.md`) chứ không
`assert` cứng một con số cụ thể — DB trên Drive có thể đã được cập nhật sau lượt
đo D-162 (16.515 chunk), và mục tiêu ở đây chỉ là "server khởi động được với DB
thật", không phải một cổng đo hồi quy.

In [ ]:
import json
import sys
from collections import Counter

sys.path.insert(0, "/content/project-bio-rag")
import chromadb

client = chromadb.PersistentClient(path=os.environ["RAG_DATABASE_DIR"])

ps = client.get_collection("processing_status")
docs = [json.loads(d) for d in ps.get(include=["documents"])["documents"]]
print("processing_status:", len(docs))
print("text_extraction_version:", Counter(d.get("text_extraction_version") for d in docs))

bt = client.get_collection("biology_text")
print("biology_text count:", bt.count())

bi = client.get_collection("biology_images")
print("biology_images count:", bi.count())

bm25_meta_path = Path(os.environ["RAG_DATABASE_DIR"]) / "sparse" / "bm25_meta.json"
if bm25_meta_path.exists():
    with open(bm25_meta_path, encoding="utf-8") as f:
        meta = json.load(f)
    print("BM25 ids:", len(meta["ids"]), "| vocab:", len(meta["vocab"]))
else:
    print("CANH BAO: khong thay", bm25_meta_path, "- RETRIEVAL_MODE=hybrid/bm25 se loi.")

assert bt.count() > 0, "biology_text rong - DB_SOURCE_DIR co the tro sai cho."
print("\nOK - DB co du lieu, chay tiep duoc.")

## 7. Giữ phiên sống

Colab ngắt phiên nếu 90 phút không tương tác với trang. Giữ tab này mở suốt buổi
demo/test FE.

In [ ]:
%%javascript
function KeepClicking(){
  var btn = document.querySelector("colab-connect-button");
  if (btn) { btn.click(); console.log("Da bam connect luc " + new Date()); }
}
setInterval(KeepClicking, 60000);

## 8. Tải `cloudflared`

Binary chính chủ Cloudflare, không cần `apt`/tài khoản. Dùng **Quick Tunnel**
(`cloudflared tunnel --url ...`) — sinh một URL ngẫu nhiên trên
`trycloudflare.com`, sống tới khi tiến trình bị dừng, không cần đăng nhập hay
đặt DNS. Đủ cho việc bật FE local test, không dùng cho production (URL không
cố định, không có SLA).

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared --version

## 9. Khởi động Flask API ở nền, đợi `/api/health`

`main.py --api` gọi `AppServices.get_instance()` (nạp bge-m3 + reranker + CLIP +
Qwen2.5-3B) TRƯỚC KHI `app.run()` mở cổng, nên `/api/health` chỉ trả 200 sau khi
mọi model đã sẵn sàng — không có khoảng "cổng mở nhưng câu hỏi đầu lỗi" (xem
docstring `health()` trong `src/app/api.py`). Chạy qua `subprocess.Popen` (nền,
không block cell) + ghi log ra file, rồi poll `/api/health` bằng `urllib`
(không phụ thuộc `requests` có sẵn hay không).

In [ ]:
import subprocess
import sys
import time
import urllib.request
import urllib.error

PORT = 5000

flask_log = open("/content/flask_api.log", "w", buffering=1)
flask_proc = subprocess.Popen(
    [sys.executable, "-u", "main.py", "--api", "--port", str(PORT)],
    cwd="/content/project-bio-rag",
    stdout=flask_log, stderr=subprocess.STDOUT,
)
print("Da khoi dong Flask API (PID", flask_proc.pid, "), dang cho model nap...")

health_url = f"http://127.0.0.1:{PORT}/api/health"
timeout_s = 600
t0 = time.time()
while True:
    if flask_proc.poll() is not None:
        with open("/content/flask_api.log") as f:
            print(f.read()[-4000:])
        raise RuntimeError(
            f"Flask API thoat som (ma thoat {flask_proc.returncode}) - xem log o tren.")
    try:
        with urllib.request.urlopen(health_url, timeout=5) as resp:
            if resp.status == 200:
                print(resp.read().decode())
                break
    except (urllib.error.URLError, ConnectionError):
        pass
    if time.time() - t0 > timeout_s:
        with open("/content/flask_api.log") as f:
            print(f.read()[-4000:])
        raise RuntimeError(f"Flask API khong san sang sau {timeout_s}s - xem log o tren.")
    time.sleep(5)
    print(f"  ... cho model nap ({int(time.time() - t0)}s)")

print("\nOK - Flask API san sang tren", health_url)

## 10. Mở tunnel Cloudflare trỏ vào cổng đó

Đọc log của `cloudflared` cho tới khi thấy dòng chứa URL `trycloudflare.com`.

In [ ]:
import re
import subprocess
import time

tunnel_log_path = "/content/cloudflared.log"
tunnel_log = open(tunnel_log_path, "w", buffering=1)
tunnel_proc = subprocess.Popen(
    ["/content/project-bio-rag/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    cwd="/content/project-bio-rag",
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)
print("Da khoi dong cloudflared (PID", tunnel_proc.pid, "), dang doi URL...")

url_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
tunnel_url = None
t0 = time.time()
while time.time() - t0 < 60:
    if tunnel_proc.poll() is not None:
        with open(tunnel_log_path) as f:
            print(f.read())
        raise RuntimeError(f"cloudflared thoat som (ma thoat {tunnel_proc.returncode}) - xem log o tren.")
    with open(tunnel_log_path) as f:
        noi_dung = f.read()
    m = url_pattern.search(noi_dung)
    if m:
        tunnel_url = m.group(0)
        break
    time.sleep(1)

if not tunnel_url:
    with open(tunnel_log_path) as f:
        print(f.read())
    raise RuntimeError("Khong tim thay URL trycloudflare.com trong log sau 60s - xem log o tren.")

print("=" * 70)
print("API DA SAN SANG O:", tunnel_url)
print("=" * 70)
print("Dan URL nay vao NEXT_PUBLIC_API_HOST cua FE local (xem o duoi).")

## 11. Trỏ FE local vào URL này

Trên máy local, ở repo `project_rag_fe`:

1. Mở (hoặc tạo) `.env.local`, đặt:
   ```
   NEXT_PUBLIC_API_HOST=<dán URL trycloudflare.com in ra ở mục 10>
   ```
2. `npm run dev` (hoặc restart nếu đã chạy — Next.js chỉ đọc `.env.local` lúc
   khởi động).
3. Ảnh hình cũng đi qua cùng URL đó (`/images/<sách>/<tệp>`, `send_from_directory`
   phía `src/app/api.py`) — không cần cấu hình gì thêm, `CORS(app)` phía server
   đã cho phép mọi origin.

**URL đổi mỗi khi mục 10 chạy lại** (Quick Tunnel không cố định) — nhớ cập nhật
lại `.env.local` sau mỗi lần restart notebook.

## 12. Giữ tunnel + server sống, theo dõi log

Chạy cell dưới và để notebook mở suốt buổi test — nó chỉ in một dòng nhịp tim
mỗi 30s và dừng NGAY nếu một trong hai tiến trình chết bất ngờ (in log kèm theo
để biết lý do). Dừng cell này bằng nút ⏹ (Interrupt) khi xong việc — server và
tunnel VẪN chạy nền sau khi Interrupt, dùng mục 13 để tắt hẳn.

In [ ]:
import time

try:
    while True:
        if flask_proc.poll() is not None:
            with open("/content/flask_api.log") as f:
                print(f.read()[-4000:])
            raise RuntimeError(f"Flask API da chet (ma thoat {flask_proc.returncode}).")
        if tunnel_proc.poll() is not None:
            with open(tunnel_log_path) as f:
                print(f.read()[-4000:])
            raise RuntimeError(f"cloudflared da chet (ma thoat {tunnel_proc.returncode}).")
        print(time.strftime("%H:%M:%S"), "- van song. URL:", tunnel_url)
        time.sleep(30)
except KeyboardInterrupt:
    print("Da dung theo doi (Interrupt) - server/tunnel VAN chay nen. Dung han qua muc 13.")

## 13. Dừng server + tunnel (chạy khi xong việc)

Không bắt buộc chạy trong lúc demo — chỉ chạy khi bạn muốn tắt hẳn trước khi
đóng notebook, để giải phóng GPU cho phiên Colab khác.

In [ ]:
for proc, ten in [(tunnel_proc, "cloudflared"), (flask_proc, "Flask API")]:
    try:
        proc.terminate()
        proc.wait(timeout=10)
        print(f"Da dung {ten} (PID {proc.pid}).")
    except Exception as exc:
        print(f"Loi khi dung {ten}: {exc}")